In [0]:
import math

from scipy.optimize import brentq  # or fsolve/newton if you prefer


In [0]:
def lmtd(T_hot_in, T_hot_out, T_cold_in, T_cold_out):
    """
    Log mean temperature difference for a generic exchanger.
    Here we'll treat the beer as 'hot' (T_hot_in ~ T_hot_out = T_b, isothermal),
    and the coolant as 'cold'.
    """
    dT1 = T_hot_in - T_cold_out   # terminal difference at one end
    dT2 = T_hot_out - T_cold_in   # terminal difference at other end

    # Guard against division by zero / identical terminal differences
    if abs(dT1 - dT2) < 1e-9:
        return dT1

    return (dT1 - dT2) / math.log(dT1 / dT2)


In [0]:
def Tc_out_steady_state(Tb, Tc_in, U, A, m_dot_c, cp_c):
    """
    Solve for coolant outlet temperature Tc_out such that
    Q_cool (from LMTD) = m_dot_c * cp_c * (Tc_out - Tc_in)
    for an isothermal fermentor at temperature Tb.
    Units must be consistent.
    """
    def residual(Tc_out):
        # LMTD with beer isothermal: T_hot_in = T_hot_out = Tb
        DeltaT_lm = lmtd(T_hot_in=Tb, T_hot_out=Tb,
                         T_cold_in=Tc_in, T_cold_out=Tc_out)

        Q_UA = U * A * DeltaT_lm
        Q_mcp = m_dot_c * cp_c * (Tc_out - Tc_in)

        return Q_mcp - Q_UA  # root when Q_mcp = Q_UA

    # Bracket for Tc_out: it must be between Tc_in and Tb (for cooling)
    Tc_low = Tc_in
    Tc_high = Tb - 1e-6  # avoid equality, which would give zero driving force

    # You can adjust the bracket if needed (e.g. for heating instead of cooling)
    Tc_out_solution = brentq(residual, Tc_low, Tc_high)
    return Tc_out_solution


In [0]:



# Example usage (replace with your values, SI units recommended):
if __name__ == "__main__":
    Tb = 19.0 + 273.15      # beer temperature [K]
    Tc_in = 15.0 + 273.15    # coolant inlet [K]
    U = 800.0               # W/(m^2 K)
    A = 1.76                 # m^2
    m_dot_c = 1055.781299           # kg/s
    cp_c = 4178.9           # J/(kg K) for water/glycol approx

    Tc_out = Tc_out_steady_state(Tb, Tc_in, U, A, m_dot_c, cp_c)
    Q = m_dot_c * cp_c * (Tc_out - Tc_in)

    print("Coolant outlet temperature [°C]:", Tc_out - 273.15)
    print("Heat removed (W) = fermentation heat load:", Q)


Coolant outlet temperature [°C]: 15.001276313626875
Heat removed (W) = fermentation heat load: 5631.101427391484
